# Junction Read Retrieval
Input a system (is_id), batch number, and circle type to retrieve junction reads.

In [ ]:
import json
import pandas as pd
import os
from pathlib import Path
import glob

OUTROOT = Path(os.environ.get("CYCLE_DATA_ROOT", "/groups/rubin/projects/kuang/out/IS_cycle"))

In [ ]:
# === INPUT ===
IS_ID = "aa620927-33c2-41e9-a7ca-33ba0cf25c27"
BATCH = 0           # batch number (0-24)
CIRCLE_TYPE = "full"  # "full" or "partial"

In [ ]:
# Find which sample contains this system
batch_name = f"batch_{BATCH:03d}"
batch_dir = OUTROOT / batch_name
fmt_dir = batch_dir / "is_formatter_output"

sample_id = None
system_info = None

for gj in sorted(fmt_dir.glob("*/*_is_records_guide.json")):
    with open(gj) as f:
        content = f.read()
    if IS_ID not in content:
        continue
    data = json.loads(content)
    for rec in data:
        if rec.get("is_id") == IS_ID:
            sample_id = rec["sample_id"]
            system_info = rec
            break
    if sample_id:
        break

if not sample_id:
    print(f"System {IS_ID} not found in {batch_name}")
else:
    print(f"System: {IS_ID}")
    print(f"Sample: {sample_id}")
    print(f"Batch:  {batch_name}")
    print(f"IS length: {len(system_info['is_element']['sequence'])}bp")
    print(f"ORFs: {system_info.get('orf_annotation', {}).get('num_orfs', '?')}")
    ce = system_info.get("circle_evidence", {})
    if ce:
        print(f"\nCircle evidence (from pipeline):")
        print(f"  tail_head (full circle):  {ce.get('n_tail_head_reads', 0)}")
        print(f"  genome_head:              {ce.get('n_genome_head_reads', 0)}")
        print(f"  tail_genome:              {ce.get('n_tail_genome_reads', 0)}")

In [ ]:
# Retrieve junction reads
if sample_id is None:
    raise ValueError("System not found — check IS_ID and BATCH above")

reads_df = None

if CIRCLE_TYPE == "full":
    # Search circle_reads.tsv in chunks (new model) or old pipeline dirs
    patterns = [
        batch_dir / "chunks" / "*" / "circle_output" / sample_id / f"{sample_id}_circle_reads.tsv",
        batch_dir / "circle_output" / sample_id / f"{sample_id}_circle_reads.tsv",
    ]
    for pat in patterns:
        matches = sorted(glob.glob(str(pat)))
        if matches:
            reads_df = pd.read_csv(matches[0], sep="\t")
            reads_df = reads_df[reads_df["is_uuid"] == IS_ID]
            print(f"Found: {matches[0]}")
            break
    if reads_df is None:
        print(f"No circle_reads.tsv found for {sample_id} in {batch_name}.")
        print("The file may have been cleaned up. Only summary counts are available (see above).")

elif CIRCLE_TYPE == "partial":
    # Search partial_circle_reads.tsv
    patterns = [
        batch_dir / "partial_circle_output" / sample_id / sample_id / f"{sample_id}_partial_circle_reads.tsv",
        batch_dir / "partial_circle_output" / sample_id / f"{sample_id}_partial_circle_reads.tsv",
    ]
    for pat in patterns:
        if Path(pat).exists():
            reads_df = pd.read_csv(str(pat), sep="\t")
            reads_df = reads_df[reads_df["is_id"] == IS_ID]
            print(f"Found: {pat}")
            break
    # Also check summary JSON for supporting read names
    for pat in [
        batch_dir / "partial_circle_output" / sample_id / sample_id / f"{sample_id}_partial_circle_summary.json",
        batch_dir / "partial_circle_output" / sample_id / f"{sample_id}_partial_circle_summary.json",
    ]:
        if Path(pat).exists():
            with open(pat) as f:
                pc_summary = json.load(f)
            if isinstance(pc_summary, list):
                pc_summary = [s for s in pc_summary if s.get("is_id") == IS_ID]
            elif isinstance(pc_summary, dict):
                pc_summary = [pc_summary] if pc_summary.get("is_id") == IS_ID else []
            if pc_summary:
                print(f"\nPartial circle summary:")
                for s in pc_summary:
                    print(f"  circle region: {s['circle_start']}-{s['circle_end']} ({s['circle_size']}bp, {s['circle_fraction']:.1%})")
                    print(f"  supporting reads: {s['n_supporting_reads']}")
                    if 'supporting_read_names' in s:
                        print(f"  read names: {s['supporting_read_names']}")
            break
    if reads_df is None:
        print(f"No partial_circle_reads.tsv found for {sample_id} in {batch_name}.")

else:
    print(f"Unknown CIRCLE_TYPE: {CIRCLE_TYPE}. Use 'full' or 'partial'.")

In [ ]:
# Display reads
if reads_df is not None and len(reads_df) > 0:
    print(f"\n{len(reads_df)} junction reads for {IS_ID}:\n")
    display(reads_df)
elif reads_df is not None:
    print(f"No reads found for {IS_ID} in the reads file (other systems may have reads).")
else:
    print("No reads data available.")